In [12]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import os
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.windows import from_bounds
from rasterio.windows import Window, from_bounds

In [7]:
import rasterio, numpy as np

path = r"C:\D\Harisankar\MY PERSONAL WORKS\sentinel2-water-quality-bathymetry\F7_2024.tif\F7_2024.tif"   # change to your path

with rasterio.open(path) as src:
    print("Bands/dtype:", src.count, src.dtypes)
    print("CRS:", src.crs)
    print("Shape:", src.shape)
    print("Bounds:", src.bounds)
    print("Res (arc-sec):", src.res[0] * 3600)
    print("Nodata:", src.nodata)

    # Read only your study box, so a large tile does not fill memory
    from rasterio.windows import from_bounds
    win = from_bounds(22.95, 35.05, 24.15, 36.25, src.transform)
    d = src.read(1, window=win).astype("float32")

print("Unique values:", len(np.unique(d)))
print("Min/max:", np.nanmin(d), np.nanmax(d))
print("Cells with depth 0-30 m (negative convention):", np.sum((d < 0) & (d >= -30)))
print("Cells with depth 0-30 m (positive convention):", np.sum((d > 0) & (d <= 30)))

Bands/dtype: 1 ('float32',)
CRS: EPSG:4326
Shape: (9004, 9484)
Bounds: BoundingBox(left=23.247916666666665, bottom=33.74791666666667, right=33.12708333333333, top=43.12708333333333)
Res (arc-sec): 3.75
Nodata: nan
Unique values: 573852
Min/max: -3825.6 1.1523438
Cells with depth 0-30 m (negative convention): 18361
Cells with depth 0-30 m (positive convention): 644


In [8]:
emod_path = r"C:\D\Harisankar\MY PERSONAL WORKS\sentinel2-water-quality-bathymetry\F7_2024.tif\F7_2024.tif"
target_tif = r"F:\data_water\processed\bands\Bands 20240117.tif"
out_path = r"F:\data_water\emod_reprojected.tif"


In [9]:
with rasterio.open(target_tif) as t:                      # Sentinel-2 grid to match
    prof = t.profile.copy()
    prof.update(count=1, dtype="float32", nodata=-32767.0)
    dst_tr, dst_crs = t.transform, t.crs

with rasterio.open(emod_path) as src:
    win = from_bounds(22.95, 35.05, 24.15, 36.25, src.transform)   # only your box
    data = src.read(1, window=win).astype("float32")
    src_tr = src.window_transform(win)

    with rasterio.open(out_path, "w", **prof) as dst:
        reproject(
            source=data, destination=rasterio.band(dst, 1),
            src_transform=src_tr, src_crs=src.crs, src_nodata=np.nan,
            dst_transform=dst_tr, dst_crs=dst_crs, dst_nodata=-32767.0,
            resampling=Resampling.nearest,                # no blending across the coast
        )

In [10]:
BLOCK = 5                                            # 5x5 EMODnet cells ~ 500 m
rows, cols = np.indices(data.shape)
block_id = ((rows // BLOCK) * 10_000 + (cols // BLOCK)).astype("float32")  # unique id per block

id_path = r"F:\data_water\emodnet_block_id.tif"
with rasterio.open(id_path, "w", **prof) as dst:     # same grid/profile as the depth raster
    reproject(
        source=block_id, destination=rasterio.band(dst, 1),
        src_transform=src_tr, src_crs="EPSG:4326", src_nodata=None,
        dst_transform=dst_tr, dst_crs=dst_crs, dst_nodata=-32767.0,
        resampling=Resampling.nearest,               # ids must never be blended
    )